In [1]:
import os
import pefile
import hashlib
from collections import defaultdict
from tqdm import tqdm

def calculate_file_hash(file_path):
    """Calculate SHA-256 hash of a file."""
    sha256 = hashlib.sha256()
    try:
        with open(file_path, "rb") as f:
            for chunk in iter(lambda: f.read(4096), b""):
                sha256.update(chunk)
        return sha256.hexdigest()
    except Exception:
        return None

def is_pe_file(file_path):
    """Check if a file is a valid PE file."""
    try:
        pefile.PE(file_path, fast_load=True)
        return True
    except Exception:
        return False

def get_pe_type(file_path):
    """Determine the type of PE file (EXE, DLL, CPL, or Other)."""
    try:
        pe = pefile.PE(file_path, fast_load=True)
        if pe.is_exe():
            return "EXE"
        elif pe.is_dll():
            return "DLL"
        elif file_path.lower().endswith('.cpl'):
            return "CPL"
        else:
            return "Other"
    except Exception:
        return None

def analyze_folder(folder_path):
    """Analyze PE files in a folder and return statistics."""
    pe_types = defaultdict(int)
    file_hashes = defaultdict(int)
    total_files = 0
    pe_files = 0
    file_sizes = []
    
    # Count files for progress bar
    all_files = []
    for root, _, files in os.walk(folder_path):
        for file in files:
            all_files.append(os.path.join(root, file))
    
    # Analyze files with progress bar
    for file_path in tqdm(all_files, desc=f"Analyzing {os.path.basename(folder_path)}"):
        total_files += 1
        
        # Check if it's a PE file
        if is_pe_file(file_path):
            pe_files += 1
            pe_type = get_pe_type(file_path)
            if pe_type:
                pe_types[pe_type] += 1
            
            # Calculate file hash for duplication check
            file_hash = calculate_file_hash(file_path)
            if file_hash:
                file_hashes[file_hash] += 1
            
            # Record file size
            try:
                file_sizes.append(os.path.getsize(file_path))
            except Exception:
                pass
    
    # Calculate duplication rate
    unique_files = len(file_hashes)
    duplicate_count = sum(count - 1 for count in file_hashes.values() if count > 1)
    duplication_rate = (duplicate_count / pe_files * 100) if pe_files > 0 else 0
    
    # Calculate average file size
    avg_file_size = sum(file_sizes) / len(file_sizes) / 1024 if file_sizes else 0  # in KB
    
    return {
        "total_files": total_files,
        "pe_files": pe_files,
        "pe_types": dict(pe_types),
        "duplication_rate": duplication_rate,
        "unique_files": unique_files,
        "avg_file_size_kb": avg_file_size
    }

def main(base_path):
    """Main function to analyze malware folders, ignoring goodware."""
    # Analyze malware folders
    malware_stats = {}
    for folder in os.listdir(base_path):
        folder_path = os.path.join(base_path, folder)
        if folder not in ["goodware", "goodware.zip"] and os.path.isdir(folder_path):
            stats = analyze_folder(folder_path)
            if stats:
                malware_stats[folder] = stats
    
    # Print results
    print("\n=== Malware Analysis ===")
    if not malware_stats:
        print("No malware folders found.")
    for folder, stats in malware_stats.items():
        print(f"\nFolder: {folder}")
        print(f"Total Files: {stats['total_files']}")
        print(f"PE Files: {stats['pe_files']}")
        print(f"PE Types: {stats['pe_types']}")
        print(f"Duplication Rate: {stats['duplication_rate']:.2f}%")
        print(f"Unique Files: {stats['unique_files']}")
        print(f"Average File Size: {stats['avg_file_size_kb']:.2f} KB")

if __name__ == "__main__":
    base_path = "/mnt/data_disk1/mabon/datacopy"
    main(base_path)

Analyzing Stefano: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 5698/5698 [01:06<00:00, 85.37it/s]
Analyzing theZoo: 0it [00:00, ?it/s]
Analyzing MLSec19: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 97/97 [00:01<00:00, 95.32it/s]
Analyzing Malshare: 0it [00:00, ?it/s]
Analyzing BluePex: 0it [00:00, ?it/s]
Analyzing evasive-it: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 152/152 [00:04<00:00, 36.15it/s]
Analyzing Honeypots: 0it [00:00, ?it/s]
Analyzing CodexGiga: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 5395/5395 [00:29<00:00, 185.95it/s]
Analyzing VxHeaven


=== Malware Analysis ===

Folder: APTs
Total Files: 223
PE Files: 178
PE Types: {'DLL': 47, 'Other': 6, 'EXE': 125}
Duplication Rate: 9.55%
Unique Files: 161
Average File Size: 774.21 KB

Folder: Stefano
Total Files: 5698
PE Files: 5698
PE Types: {'EXE': 5698}
Duplication Rate: 0.00%
Unique Files: 5698
Average File Size: 1996.33 KB

Folder: theZoo
Total Files: 0
PE Files: 0
PE Types: {}
Duplication Rate: 0.00%
Unique Files: 0
Average File Size: 0.00 KB

Folder: MLSec19
Total Files: 97
PE Files: 97
PE Types: {'EXE': 97}
Duplication Rate: 0.00%
Unique Files: 97
Average File Size: 1296.03 KB

Folder: Malshare
Total Files: 0
PE Files: 0
PE Types: {}
Duplication Rate: 0.00%
Unique Files: 0
Average File Size: 0.00 KB

Folder: BluePex
Total Files: 0
PE Files: 0
PE Types: {}
Duplication Rate: 0.00%
Unique Files: 0
Average File Size: 0.00 KB

Folder: MLSec21
Total Files: 274
PE Files: 274
PE Types: {'DLL': 208, 'EXE': 66}
Duplication Rate: 28.83%
Unique Files: 195
Average File Size: 658.37 KB


In [2]:
import os
import pefile
import hashlib
from collections import defaultdict
from tqdm import tqdm

def calculate_file_hash(file_path):
    """Calculate SHA-256 hash of a file."""
    sha256 = hashlib.sha256()
    try:
        with open(file_path, "rb") as f:
            for chunk in iter(lambda: f.read(4096), b""):
                sha256.update(chunk)
        return sha256.hexdigest()
    except Exception:
        return None

def is_pe_file(file_path):
    """Check if a file is a valid PE file."""
    try:
        pefile.PE(file_path, fast_load=True)
        return True
    except Exception:
        return False

def get_pe_type(file_path):
    """Determine the type of PE file (EXE, DLL, CPL, or Other)."""
    try:
        pe = pefile.PE(file_path, fast_load=True)
        if pe.is_exe():
            return "EXE"
        elif pe.is_dll():
            return "DLL"
        elif file_path.lower().endswith('.cpl'):
            return "CPL"
        else:
            return "Other"
    except Exception:
        return None

def analyze_dataset(base_path):
    """Analyze PE files across all malware folders and return aggregated statistics."""
    pe_types = defaultdict(int)
    file_hashes = defaultdict(int)
    total_files = 0
    pe_files = 0
    file_sizes = []
    
    # Collect all files from malware folders
    all_files = []
    malware_folders = [f for f in os.listdir(base_path) if f not in ["goodware", "goodware.zip"] and os.path.isdir(os.path.join(base_path, f))]
    
    for folder in malware_folders:
        folder_path = os.path.join(base_path, folder)
        for root, _, files in os.walk(folder_path):
            for file in files:
                all_files.append(os.path.join(root, file))
    
    # Analyze files with progress bar
    for file_path in tqdm(all_files, desc="Analyzing dataset"):
        total_files += 1
        
        # Check if it's a PE file
        if is_pe_file(file_path):
            pe_files += 1
            pe_type = get_pe_type(file_path)
            if pe_type:
                pe_types[pe_type] += 1
            
            # Calculate file hash for duplication check
            file_hash = calculate_file_hash(file_path)
            if file_hash:
                file_hashes[file_hash] += 1
            
            # Record file size
            try:
                file_sizes.append(os.path.getsize(file_path))
            except Exception:
                pass
    
    # Calculate duplication rate
    unique_files = len(file_hashes)
    duplicate_count = sum(count - 1 for count in file_hashes.values() if count > 1)
    duplication_rate = (duplicate_count / pe_files * 100) if pe_files > 0 else 0
    
    # Calculate average file size
    avg_file_size = sum(file_sizes) / len(file_sizes) / 1024 if file_sizes else 0  # in KB
    
    return {
        "total_files": total_files,
        "pe_files": pe_files,
        "pe_types": dict(pe_types),
        "duplication_rate": duplication_rate,
        "unique_files": unique_files,
        "avg_file_size_kb": avg_file_size,
        "folders_analyzed": len(malware_folders)
    }

def main(base_path):
    """Main function to analyze all malware folders as a single dataset."""
    # Analyze the entire dataset
    stats = analyze_dataset(base_path)
    
    # Print results
    print("\n=== Malware Dataset Analysis ===")
    if stats["total_files"] == 0:
        print("No malware files found.")
    else:
        print(f"Folders Analyzed: {stats['folders_analyzed']}")
        print(f"Total Files: {stats['total_files']}")
        print(f"PE Files: {stats['pe_files']}")
        print(f"PE Types: {stats['pe_types']}")
        print(f"Duplication Rate: {stats['duplication_rate']:.2f}%")
        print(f"Unique Files: {stats['unique_files']}")
        print(f"Average File Size: {stats['avg_file_size_kb']:.2f} KB")

if __name__ == "__main__":
    base_path = "/mnt/data_disk1/mabon/datacopy"
    main(base_path)

Analyzing dataset: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 77151/77151 [16:06<00:00, 79.84it/s]


=== Malware Dataset Analysis ===
Folders Analyzed: 15
Total Files: 77151
PE Files: 71649
PE Types: {'DLL': 15964, 'Other': 292, 'EXE': 55393}
Duplication Rate: 15.96%
Unique Files: 60212
Average File Size: 2181.03 KB


In [3]:
import os
import pefile
import hashlib
from collections import defaultdict
from tqdm import tqdm

def calculate_file_hash(file_path):
    """Calculate SHA-256 hash of a file."""
    sha256 = hashlib.sha256()
    try:
        with open(file_path, "rb") as f:
            for chunk in iter(lambda: f.read(4096), b""):
                sha256.update(chunk)
        return sha256.hexdigest()
    except Exception:
        return None

def is_pe_file(file_path):
    """Check if a file is a valid PE file."""
    try:
        pefile.PE(file_path, fast_load=True)
        return True
    except Exception:
        return False

def get_pe_type(file_path):
    """Determine the type of PE file (EXE, DLL, CPL, or Other)."""
    try:
        pe = pefile.PE(file_path, fast_load=True)
        if pe.is_exe():
            return "EXE"
        elif pe.is_dll():
            return "DLL"
        elif file_path.lower().endswith('.cpl'):
            return "CPL"
        else:
            return "Other"
    except Exception:
        return None

def analyze_dataset(base_path):
    """Analyze PE files across all malware folders and return aggregated statistics."""
    pe_types = defaultdict(int)
    file_hashes = defaultdict(int)
    total_files = 0
    pe_files = 0
    file_sizes = []
    
    # Collect all files from malware folders
    all_files = []
    malware_folders = [f for f in os.listdir(base_path) if f not in ["goodware", "goodware.zip"] and os.path.isdir(os.path.join(base_path, f))]
    
    for folder in malware_folders:
        folder_path = os.path.join(base_path, folder)
        for root, _, files in os.walk(folder_path):
            for file in files:
                all_files.append(os.path.join(root, file))
    
    # Analyze files with progress bar
    for file_path in tqdm(all_files, desc="Analyzing dataset"):
        total_files += 1
        
        # Check if it's a PE file
        if is_pe_file(file_path):
            pe_files += 1
            pe_type = get_pe_type(file_path)
            if pe_type:
                pe_types[pe_type] += 1
            
            # Calculate file hash for duplication check
            file_hash = calculate_file_hash(file_path)
            if file_hash:
                file_hashes[file_hash] += 1
            
            # Record file size
            try:
                file_sizes.append(os.path.getsize(file_path))
            except Exception:
                pass
    
    # Calculate duplication rate
    unique_files = len(file_hashes)
    duplicate_count = sum(count - 1 for count in file_hashes.values() if count > 1)
    duplication_rate = (duplicate_count / pe_files * 100) if pe_files > 0 else 0
    
    # Calculate average file size
    avg_file_size = sum(file_sizes) / len(file_sizes) / 1024 if file_sizes else 0  # in KB
    
    return {
        "total_files": total_files,
        "pe_files": pe_files,
        "pe_types": dict(pe_types),
        "duplication_rate": duplication_rate,
        "unique_files": unique_files,
        "avg_file_size_kb": avg_file_size,
        "folders_analyzed": len(malware_folders)
    }

def main(base_path):
    """Main function to analyze all malware folders as a single dataset."""
    # Analyze the entire dataset
    stats = analyze_dataset(base_path)
    
    # Print results
    print("\n=== Malware Dataset Analysis ===")
    if stats["total_files"] == 0:
        print("No malware files found.")
    else:
        print(f"Folders Analyzed: {stats['folders_analyzed']}")
        print(f"Total Files: {stats['total_files']}")
        print(f"PE Files: {stats['pe_files']}")
        print(f"PE Types: {stats['pe_types']}")
        print(f"Duplication Rate: {stats['duplication_rate']:.2f}%")
        print(f"Unique Files: {stats['unique_files']}")
        print(f"Average File Size: {stats['avg_file_size_kb']:.2f} KB")

if __name__ == "__main__":
    base_path = "/mnt/data_disk1/mabon/goodware/"
    main(base_path)

Analyzing dataset: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 22868/22868 [13:04<00:00, 29.13it/s]


=== Malware Dataset Analysis ===
Folders Analyzed: 6
Total Files: 22868
PE Files: 22868
PE Types: {'EXE': 19304, 'DLL': 3561, 'Other': 3}
Duplication Rate: 0.76%
Unique Files: 22695
Average File Size: 10710.86 KB
